Historical plotting notebook. Start Jupyter from the repository root; use the experiment READMEs and command-line runners to generate inputs first. Cells retain exploratory plotting variants; Run All is not the reproduction entry point.

In [ ]:
from pathlib import Path
import os, sys
while not Path("pyproject.toml").exists():
    if Path.cwd() == Path.cwd().parent: raise RuntimeError("Start Jupyter inside this repository")
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
from torchvision import transforms
from torchvision.datasets import MNIST, CIFAR10
from torch.utils.data import DataLoader
from src.models.mlp import PlainNN


In [ ]:

# Create output directory for experiment 5
output_dir = "exp3_plots"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Set global plotting style parameters for consistency
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 16

In [ ]:

def get_penultimate_features(model, x, device='cuda'):
    """
    Extract penultimate layer features from the model.
    """
    model.eval()
    with torch.no_grad():
        features = model.extract_features(x)
        # Get the penultimate layer features (second to last)
        return features[-2]

def compute_jacobian(model, x, output_layer_idx=-2):
    """Compute Jacobian of penultimate layer features w.r.t. input."""
    x.requires_grad_(True)
    
    # Use the extract_features method
    features = model.extract_features(x)
    
    # Get penultimate features
    penultimate_features = features[output_layer_idx]
    
    # Get feature dimensions
    batch_size, feature_dim = penultimate_features.shape
    input_dim = x.numel()
    
    # Initialize Jacobian matrix
    jacobian = torch.zeros(feature_dim, input_dim, device=x.device)
    
    # Compute Jacobian one feature at a time
    for i in range(feature_dim):
        model.zero_grad()
        
        # Set all gradients to zero except for the current feature
        if i > 0:
            x.grad.zero_()
        
        # Set up a "fake" loss that's just the value of this feature
        penultimate_features[:, i].backward(retain_graph=(i < feature_dim-1))
        
        # Store the gradient
        jacobian[i] = x.grad.view(-1)
    
    # Reset requires_grad
    x.requires_grad_(False)
    
    return jacobian

def compute_svd(jacobian):
    """Compute SVD of the Jacobian."""
    J_np = jacobian.detach().cpu().numpy()
    U, S, Vt = np.linalg.svd(J_np, full_matrices=True)
    V = Vt.T  # So V[:, i] is the i-th right singular vector
    
    return U, S, V

def compute_linear_approximation_thresholds(model, sample, V, singular_values, cosine_threshold=0.95, 
                                           device='cuda', max_alpha=20.0, num_steps=100, 
                                           num_rsv_to_analyze=50, rsv_scale = 56.0):
    """
    Compute the alpha threshold where cosine similarity drops below a specified value.
    
    Args:
        model: Neural network model
        sample: Input sample tensor
        V: Right singular vectors from SVD
        singular_values: Singular values from SVD
        cosine_threshold: Threshold for cosine similarity (default: 0.95)
        device: Computation device
        max_alpha: Maximum perturbation magnitude
        num_steps: Number of alpha values to evaluate
        num_rsv_to_analyze: Number of RSVs to analyze (evenly distributed from highest to lowest)
        
    Returns:
        rsv_indices: Array of RSV indices that were analyzed
        thresholds: Array of alpha thresholds corresponding to the RSV indices
        sv_values: Array of singular values corresponding to the RSV indices
    """
    model.eval()
    model = model.to(device)
    
    # Ensure sample has correct dimensions
    if sample.dim() == 1:  # For flattened samples
        # Assuming MNIST format (28x28)
        sample = sample.reshape(1, 28, 28)
    
    # Move sample to device
    sample = sample.to(device)
    
    # Get original features
    orig_features = get_penultimate_features(model, sample.unsqueeze(0), device)
    
    # Get input shape for reshaping RSVs
    if sample.dim() == 3:
        C, H, W = sample.shape
    else:
        H, W = sample.shape[-2:]
        C = 1 if sample.dim() == 2 else sample.shape[0]
    
    original_shape = (C, H, W)
    
    # Select RSV indices (evenly distributed from 0 to num_rsv-1)
    num_rsv = V.shape[1]
    total_indices = min(num_rsv, num_rsv_to_analyze)
    rsv_indices = np.linspace(0, num_rsv-1, total_indices, dtype=int)
    
    # Compute Jacobian (we need this for linear approximation)
    # Prepare input for Jacobian computation
    x = sample.clone().unsqueeze(0)
    x.requires_grad_(True)
    
    # Forward pass to get features
    features = model.extract_features(x)
    penultimate_features = features[-2]
    
    # Initialize Jacobian matrix
    feature_dim = penultimate_features.shape[1]
    input_dim = x.view(1, -1).shape[1]
    jacobian = torch.zeros(feature_dim, input_dim, device=device)
    
    # Compute Jacobian one feature at a time
    for i in range(feature_dim):
        model.zero_grad()
        
        # Clear previous gradients if any
        if i > 0:
            x.grad.zero_()
        
        # Backpropagate with respect to the i-th feature
        penultimate_features[0, i].backward(retain_graph=(i < feature_dim-1))
        
        # Store the gradient
        jacobian[i] = x.grad.view(-1)
    
    # Reset requires_grad
    x.requires_grad_(False)
    
    # Prepare alpha values
    alphas = np.linspace(0, max_alpha, num_steps)
    
    # Lists to store thresholds and singular values
    thresholds = []
    sv_values = []
    
    # For each selected RSV, compute threshold
    for idx in rsv_indices: #tqdm(rsv_indices, desc="Computing thresholds"):
        # Get the right singular vector
        rsv = V[:, idx]
        rsv_norm = np.linalg.norm(rsv)
        
        # Get singular value
        sv_value = singular_values[idx] if idx < len(singular_values) else 0
        sv_values.append(sv_value)
        
        if rsv_norm > 0:
            # Normalize and reshape RSV to match input dimensions
            rsv = rsv / rsv_norm * rsv_scale  # Scale factor to maintain consistent perturbation magnitude
            rsv_reshaped = rsv.reshape(original_shape)
            rsv_tensor = torch.tensor(rsv_reshaped, dtype=torch.float32, device=device)
            
            # Compute cosine similarity at different alpha values
            threshold_found = False
            
            for alpha in alphas:
                # Create perturbed input
                perturbed_input = sample.clone() + alpha * rsv_tensor
                perturbed_input = perturbed_input.unsqueeze(0)  # Add batch dimension
                
                # Get actual features for perturbed input
                with torch.no_grad():
                    actual_features = get_penultimate_features(model, perturbed_input, device)
                
                # Compute linear approximation
                jv = torch.matmul(jacobian, torch.tensor(rsv.flatten(), device=device, dtype=torch.float32))
                linear_approx = orig_features.clone() + alpha * jv.unsqueeze(0)
                
                # Compute cosine similarity
                cos_sim = torch.nn.functional.cosine_similarity(
                    actual_features.view(-1), linear_approx.view(-1), dim=0
                ).item()
                
                # Check if similarity dropped below threshold
                if cos_sim < cosine_threshold:
                    thresholds.append(alpha)
                    threshold_found = True
                    break
            
            # If no threshold found (cosine similarity always above threshold)
            if not threshold_found:
                thresholds.append(max_alpha)
        else:
            # RSV has zero norm, unlikely but handle anyway
            thresholds.append(max_alpha)
    
    return rsv_indices, np.array(thresholds), np.array(sv_values)

def load_model(checkpoint_path, hidden_dim1, hidden_dim2, input_dim=784, output_dim=10, device='cpu'):
    # Define the PlainNN class
    
    model = PlainNN(input_dim=input_dim, hidden_dim1=hidden_dim1, 
                   hidden_dim2=hidden_dim2, output_dim=output_dim).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model

def plot_thresholds_by_rsv(rsv_indices, random_thresholds, trained_thresholds, 
                          random_sv, trained_sv, output_path='rsv_threshold_comparison', 
                          window_size=15, figsize=(10, 6)):
    """
    Plot alpha thresholds for all RSVs, comparing random and trained models.
    Include singular values on the same plot with a secondary y-axis.
    
    Args:
        rsv_indices: Indices of RSVs that were analyzed
        random_thresholds: Alpha thresholds for the random model
        trained_thresholds: Alpha thresholds for the trained model
        random_sv: Singular values for the random model
        trained_sv: Singular values for the trained model
        output_path: Path to save the output figure (without extension)
        window_size: Window size for moving average smoothing
        figsize: Figure size as tuple (width, height) in inches
    """
    from scipy.signal import savgol_filter
    
    # Create a single figure with one plot
    fig, ax1 = plt.subplots(figsize=figsize)
    
    # Apply Savitzky-Golay filter for smoothing (if enough data points)
    if len(rsv_indices) >= window_size:
        # Make sure window_size is odd
        if window_size % 2 == 0:
            window_size += 1
            
        # Use polynomial order 3 (or less if window is small)
        poly_order = min(3, window_size - 1)
        
        # Apply filter
        random_smooth = savgol_filter(random_thresholds, window_size, poly_order)
        trained_smooth = savgol_filter(trained_thresholds, window_size, poly_order)
        
        # Plot smoothed lines (thicker and more opaque)
        ax1.plot(rsv_indices, random_smooth, color='blue', linewidth=2.5, 
                label='Untrained Model α', alpha=0.8)
        ax1.plot(rsv_indices, trained_smooth, color='red', linewidth=2.5, 
                label='Trained Model α', alpha=0.8)
    else:
        # If not enough points for Savitzky-Golay, use simple moving average
        def moving_average(data, window_size):
            cumsum = np.cumsum(np.insert(data, 0, 0)) 
            return (cumsum[window_size:] - cumsum[:-window_size]) / window_size
        
        # Use smaller window if necessary
        ma_window = min(window_size // 2, len(rsv_indices) // 2)
        if ma_window >= 2:
            # Compute moving average
            ma_random = moving_average(random_thresholds, ma_window)
            ma_trained = moving_average(trained_thresholds, ma_window)
            
            # Convert to full length for plotting (pad ends)
            pad = ma_window // 2
            full_ma_random = np.pad(ma_random, (pad, pad), 'edge')[:len(random_thresholds)]
            full_ma_trained = np.pad(ma_trained, (pad, pad), 'edge')[:len(trained_thresholds)]
            
            # Plot moving average
            ax1.plot(rsv_indices, full_ma_random, color='blue', linewidth=2.5, 
                    label='Untrained Model α', alpha=0.8)
            ax1.plot(rsv_indices, full_ma_trained, color='red', linewidth=2.5, 
                    label='Trained Model α', alpha=0.8)
    
    ax1.set_xlabel("RSV Index (ordered by singular value)")
    ax1.set_ylabel("Max α with Cosine Similarity ≥ 0.95", color='black')
    ax1.tick_params(axis='y', labelcolor='black')
    ax1.grid(alpha=0.3)
    
    # Create secondary y-axis for singular values
    ax2 = ax1.twinx()
    
    # Plot singular values on secondary y-axis (using dashed lines)
    ax2.semilogy(rsv_indices, random_sv, color='blue', alpha=0.5, label='Untrained Model σ', 
                linestyle='--', marker=None)
    ax2.semilogy(rsv_indices, trained_sv, color='red', alpha=0.5, label='Trained Model σ', 
                linestyle='--', marker=None)
    
    ax2.set_ylabel("Singular Value (log scale)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    
    # Combine legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
    
    # Set title
    plt.title("Linear Approximation Thresholds and Singular Values across RSV Spectrum")
    
    plt.tight_layout()
    
    # Save as PNG and PDF
    plt.savefig(f"{output_path}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_path}.pdf", format='pdf', bbox_inches='tight')
    
    plt.show()



In [ ]:
def analyze_multiple_samples(random_checkpoint, trained_checkpoint, hidden_dim1, hidden_dim2, 
                            data_path, output_dir="exp3_plots", device='cuda', num_rsv_to_analyze=100, 
                            samples_per_class=5, class_count=10, figsize=(14, 8)):
    """
    Analyze linear approximation thresholds for multiple samples from each class.
    
    Args:
        random_checkpoint: Path to random initialization checkpoint
        trained_checkpoint: Path to trained model checkpoint
        hidden_dim1: Hidden dimension 1
        hidden_dim2: Hidden dimension 2
        data_path: Path to dataset
        output_dir: Directory to save output plots
        device: Device to run computations on
        num_rsv_to_analyze: Number of RSVs to analyze
        samples_per_class: Number of samples to analyze per class
        class_count: Number of classes in dataset
        figsize: Figure size for plots
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load models
    random_model = load_model(random_checkpoint, hidden_dim1, hidden_dim2, device=device)
    trained_model = load_model(trained_checkpoint, hidden_dim1, hidden_dim2, device=device)
    
    # Load dataset
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    train_dataset = MNIST(root=data_path, train=True, transform=transform, download=True)
    
    # Create DataLoader to iterate through dataset
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)
    
    # Dictionary to store samples for each class
    class_samples = {i: [] for i in range(class_count)}
    
    # Collect samples for each class
    print("Collecting samples for each class...")
    for image, label in train_loader:
        label = label.item()
        if len(class_samples[label]) < samples_per_class:
            class_samples[label].append(image.squeeze(0))
        
        # Check if we have enough samples for all classes
        if all(len(samples) >= samples_per_class for samples in class_samples.values()):
            break
    
    # Initialize result storage
    results = {
        'rsv_indices': None,
        'random_model': {
            'thresholds_by_class': {i: [] for i in range(class_count)},
            'singular_values_by_class': {i: [] for i in range(class_count)},
            'avg_thresholds': None,
            'avg_singular_values': None
        },
        'trained_model': {
            'thresholds_by_class': {i: [] for i in range(class_count)},
            'singular_values_by_class': {i: [] for i in range(class_count)},
            'avg_thresholds': None,
            'avg_singular_values': None
        }
    }
    
    # Analyze each class separately
    for class_idx in range(class_count):
        print(f"\nAnalyzing class {class_idx}...")
        
        # Analyze each sample for this class
        for sample_idx, sample in enumerate(class_samples[class_idx]):
            print(f"  Processing sample {sample_idx+1}/{samples_per_class} of class {class_idx}...")
            
            # Compute Jacobian for the sample
            jacobian_random = compute_jacobian(random_model, sample.unsqueeze(0).to(device))
            jacobian_trained = compute_jacobian(trained_model, sample.unsqueeze(0).to(device))
            
            # Compute SVD
            _, S_random, V_random = compute_svd(jacobian_random)
            _, S_trained, V_trained = compute_svd(jacobian_trained)
            
            # Compute thresholds
            rsv_indices_random, thresholds_random, sv_values_random = compute_linear_approximation_thresholds(
                random_model, sample, V_random, S_random, 
                device=device, num_rsv_to_analyze=num_rsv_to_analyze
            )
            
            rsv_indices_trained, thresholds_trained, sv_values_trained = compute_linear_approximation_thresholds(
                trained_model, sample, V_trained, S_trained, 
                device=device, num_rsv_to_analyze=num_rsv_to_analyze
            )
            
            # Store results
            if results['rsv_indices'] is None:
                results['rsv_indices'] = rsv_indices_random  # Should be the same for all samples
            
            results['random_model']['thresholds_by_class'][class_idx].append(thresholds_random)
            results['random_model']['singular_values_by_class'][class_idx].append(sv_values_random)
            
            results['trained_model']['thresholds_by_class'][class_idx].append(thresholds_trained)
            results['trained_model']['singular_values_by_class'][class_idx].append(sv_values_trained)
    
    # Compute averages for each class
    random_avg_thresholds_by_class = {}
    random_avg_sv_by_class = {}
    trained_avg_thresholds_by_class = {}
    trained_avg_sv_by_class = {}
    
    for class_idx in range(class_count):
        # Stack all samples for this class
        random_thresholds_stack = np.stack(results['random_model']['thresholds_by_class'][class_idx])
        random_sv_stack = np.stack(results['random_model']['singular_values_by_class'][class_idx])
        
        trained_thresholds_stack = np.stack(results['trained_model']['thresholds_by_class'][class_idx])
        trained_sv_stack = np.stack(results['trained_model']['singular_values_by_class'][class_idx])
        
        # Compute averages
        random_avg_thresholds_by_class[class_idx] = np.mean(random_thresholds_stack, axis=0)
        random_avg_sv_by_class[class_idx] = np.mean(random_sv_stack, axis=0)
        
        trained_avg_thresholds_by_class[class_idx] = np.mean(trained_thresholds_stack, axis=0)
        trained_avg_sv_by_class[class_idx] = np.mean(trained_sv_stack, axis=0)
    
    # Compute overall averages across all classes
    random_all_thresholds = np.stack([random_avg_thresholds_by_class[i] for i in range(class_count)])
    random_all_sv = np.stack([random_avg_sv_by_class[i] for i in range(class_count)])
    
    trained_all_thresholds = np.stack([trained_avg_thresholds_by_class[i] for i in range(class_count)])
    trained_all_sv = np.stack([trained_avg_sv_by_class[i] for i in range(class_count)])
    
    results['random_model']['avg_thresholds'] = np.mean(random_all_thresholds, axis=0)
    results['random_model']['avg_singular_values'] = np.mean(random_all_sv, axis=0)
    
    results['trained_model']['avg_thresholds'] = np.mean(trained_all_thresholds, axis=0)
    results['trained_model']['avg_singular_values'] = np.mean(trained_all_sv, axis=0)
    
    # Save all the per-class averages too
    results['random_model']['avg_thresholds_by_class'] = random_avg_thresholds_by_class
    results['random_model']['avg_singular_values_by_class'] = random_avg_sv_by_class
    results['trained_model']['avg_thresholds_by_class'] = trained_avg_thresholds_by_class
    results['trained_model']['avg_singular_values_by_class'] = trained_avg_sv_by_class
    
    # Plot the average results across all classes
    plot_thresholds_by_rsv(
        results['rsv_indices'], 
        results['random_model']['avg_thresholds'], 
        results['trained_model']['avg_thresholds'], 
        results['random_model']['avg_singular_values'], 
        results['trained_model']['avg_singular_values'],
        output_path=os.path.join(output_dir, 'rsv_threshold_comparison_all_classes'),
        figsize=figsize
    )
    
    # Also plot individual class results
    for class_idx in range(class_count):
        plot_thresholds_by_rsv(
            results['rsv_indices'], 
            random_avg_thresholds_by_class[class_idx], 
            trained_avg_thresholds_by_class[class_idx], 
            random_avg_sv_by_class[class_idx], 
            trained_avg_sv_by_class[class_idx],
            output_path=os.path.join(output_dir, f'rsv_threshold_comparison_class_{class_idx}'),
            figsize=figsize
        )
    
    return results

In [ ]:
# Define paths and parameters
random_checkpoint = "checkpoints/mnist_original_mlp/model_epoch0_batch0.pth"
trained_checkpoint = "checkpoints/mnist_original_mlp/model_epoch9_batch399.pth"
data_path = "../data/MNIST-Original"
hidden_dim1 = 1024
hidden_dim2 = 512

# Run multi-sample, multi-class analysis
results = analyze_multiple_samples(
    random_checkpoint=random_checkpoint,
    trained_checkpoint=trained_checkpoint,
    hidden_dim1=hidden_dim1,
    hidden_dim2=hidden_dim2,
    data_path=data_path,
    device="cuda",
    num_rsv_to_analyze=100,
    samples_per_class=5,  # Analyze 5 samples per class
    class_count=10        # 10 classes for MNIST
)

In [ ]:
def plot_multiclass_comparison(results, output_path='multiclass_comparison', 
                             class_ids=None, window_size=15, class_opacity=0.4,
                             figsize=(10, 6)):  # Added figsize parameter
    """
    Plot comparison of multiple classes' thresholds on the same graph.
    
    Args:
        results: Results dictionary from analyze_multiple_samples
        output_path: Path to save the output figure (without extension)
        class_ids: List of class IDs to plot (None = all classes)
        window_size: Window size for moving average smoothing
        class_opacity: Opacity level for individual class lines (0.0-1.0)
        figsize: Figure size as tuple (width, height) in inches
    """
    from scipy.signal import savgol_filter
    
    # If no specific classes provided, use all classes
    if class_ids is None:
        class_ids = list(results['trained_model']['avg_thresholds_by_class'].keys())
    
    # Create figure with specified size (overriding the global setting)
    fig, ax1 = plt.subplots(figsize=figsize)
    
    # Colors for different classes
    cmap = plt.get_cmap('tab10')
    
    # Plot trained model thresholds for each class
    for i, class_idx in enumerate(class_ids):
        color = cmap(i % 10)
        thresholds = results['trained_model']['avg_thresholds_by_class'][class_idx]
        rsv_indices = results['rsv_indices']
        
        # Apply smoothing if enough data points
        if len(rsv_indices) >= window_size:
            if window_size % 2 == 0:
                window_size += 1
            
            poly_order = min(3, window_size - 1)
            smooth = savgol_filter(thresholds, window_size, poly_order)
            
            # Plot smoothed line with custom opacity
            ax1.plot(rsv_indices, smooth, color=color, linewidth=2.5, 
                    label=f'Class {class_idx}', alpha=class_opacity)
        else:
            # Simple moving average for small datasets
            def moving_average(data, window_size):
                cumsum = np.cumsum(np.insert(data, 0, 0)) 
                return (cumsum[window_size:] - cumsum[:-window_size]) / window_size
            
            ma_window = min(window_size // 2, len(rsv_indices) // 2)
            if ma_window >= 2:
                ma = moving_average(thresholds, ma_window)
                pad = ma_window // 2
                full_ma = np.pad(ma, (pad, pad), 'edge')[:len(thresholds)]
                ax1.plot(rsv_indices, full_ma, color=color, linewidth=2.5, 
                        label=f'Class {class_idx}', alpha=class_opacity)
    
    # Add overall average as a black line
    overall_avg = results['trained_model']['avg_thresholds']
    ax1.plot(rsv_indices, overall_avg, color='black', linewidth=3.0, 
             label='Trained Model Average α (All Classes)', alpha=1.0, linestyle='-')
    
    # Add overall average from random model for comparison
    random_overall = results['random_model']['avg_thresholds']
    ax1.plot(rsv_indices, random_overall, color='gray', linewidth=3.0, 
             label='Untrained Model Average α (All Classes)', alpha=0.7, linestyle='--')
    
    # Set labels and title
    ax1.set_xlabel("RSV Index (ordered by singular value)")
    ax1.set_ylabel("Max α with Cosine Similarity ≥ 0.95")
    ax1.grid(alpha=0.3)
    
    # Create secondary y-axis for singular values
    ax2 = ax1.twinx()
    
    # Plot average singular values for both models on secondary y-axis
    ax2.semilogy(rsv_indices, results['trained_model']['avg_singular_values'], 
                color='black', alpha=0.5, label='Trained Model Avg Singular Values (σ)', 
                linestyle=':', linewidth=2.0)
    
    ax2.semilogy(rsv_indices, results['random_model']['avg_singular_values'], 
                color='red', alpha=0.5, label='Untrained Model Avg Singular Values (σ)', 
                linestyle='-.', linewidth=2.0)
    
    ax2.set_ylabel("Singular Value (log scale)")
    
    # Combine legends
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', 
              ncol=2 if len(class_ids) > 5 else 1)
    
    plt.title("Comparison of Linear Approximation Thresholds Across Classes")
    plt.tight_layout()
    
    # Save as PNG
    plt.savefig(f"{output_path}.png", dpi=300, bbox_inches='tight')
    
    # Save as PDF
    plt.savefig(f"{output_path}.pdf", format='pdf', bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_multiclass_comparison(results, 
                          output_path=os.path.join(output_dir, 'multiclass_comparison'),
                          class_opacity=0.7, figsize=(14,8))

In [ ]:

import numpy as np
import pickle
import os
import matplotlib.pyplot as plt

def save_results(results, filename):
    """
    Save results dictionary to a pickle file.
    
    Args:
        results: Results dictionary from analyze_multiple_samples
        filename: Path where to save the results (will add .pkl extension if not present)
    """
    # Add .pkl extension if not present
    if not filename.endswith('.pkl'):
        filename += '.pkl'
    
    # Save the results dictionary
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    
    print(f"Results saved to {filename}")

def load_results(filename):
    """
    Load results dictionary from a pickle file.
    
    Args:
        filename: Path to the saved results file (.pkl extension will be added if not present)
        
    Returns:
        The loaded results dictionary
    """
    # Add .pkl extension if not present
    if not filename.endswith('.pkl'):
        filename += '.pkl'
    
    # Check if file exists
    if not os.path.exists(filename):
        raise FileNotFoundError(f"File {filename} not found.")
    
    # Load the results dictionary
    with open(filename, 'rb') as f:
        results = pickle.load(f)
    
    print(f"Results loaded from {filename}")
    return results

# RUBBISH CODE

In [ ]:
def plot_multiclass_comparison(results, output_path='multiclass_comparison.png', 
                              class_ids=None, window_size=15, 
                              show_class_lines=True, show_raw_data=False,
                              class_line_opacity=0.4):
    """
    Plot comparison of multiple classes' thresholds on the same graph.
    
    Args:
        results: Results dictionary from analyze_multiple_samples
        output_path: Path to save the output figure
        class_ids: List of class IDs to plot (None = all classes)
        window_size: Window size for moving average smoothing
        show_class_lines: Whether to show individual class lines (True) or only overall averages (False)
        show_raw_data: Whether to show raw data points (True) or only smoothed lines (False)
        class_line_opacity: Opacity level for class-specific lines (0.0 to 1.0)
    """
    from scipy.signal import savgol_filter
    
    # Set font sizes
    SMALL_SIZE = 20    # Small text: tick labels
    MEDIUM_SIZE = 22   # Medium text: axes labels, legend
    BIGGER_SIZE = 24   # Bigger text: title
    
    # Update font sizes
    plt.rc('font', size=SMALL_SIZE)          # Default text sizes
    plt.rc('axes', titlesize=MEDIUM_SIZE)    # Axes title
    plt.rc('axes', labelsize=MEDIUM_SIZE)    # X and Y labels
    plt.rc('xtick', labelsize=SMALL_SIZE)    # X tick labels
    plt.rc('ytick', labelsize=SMALL_SIZE)    # Y tick labels
    plt.rc('legend', fontsize=MEDIUM_SIZE)   # Legend
    plt.rc('figure', titlesize=BIGGER_SIZE)  # Figure title
    
    # If no specific classes provided, use all classes
    if class_ids is None:
        class_ids = list(results['trained_model']['avg_thresholds_by_class'].keys())
    
    # Create figure
    fig, ax1 = plt.subplots(figsize=(16, 10))
    
    # Colors for different classes
    cmap = plt.get_cmap('tab10')
    
    # Get RSV indices
    rsv_indices = results['rsv_indices']
    
    # Plot trained model thresholds for each class if requested
    if show_class_lines:
        for i, class_idx in enumerate(class_ids):
            color = cmap(i % 10)
            thresholds = results['trained_model']['avg_thresholds_by_class'][class_idx]
            
            # Plot raw data if requested
            if show_raw_data:
                ax1.scatter(rsv_indices, thresholds, color=color, marker='o', s=15, alpha=class_line_opacity*0.5, 
                           label=f'Class {class_idx}')
            
            # Apply smoothing if enough data points
            if len(rsv_indices) >= window_size:
                if window_size % 2 == 0:
                    window_size += 1
                
                poly_order = min(3, window_size - 1)
                smooth = savgol_filter(thresholds, window_size, poly_order)
                
                # Plot smoothed line with reduced opacity
                ax1.plot(rsv_indices, smooth, color=color, linewidth=2.0, 
                        label=f'Class {class_idx}', alpha=class_line_opacity)
            else:
                # Simple moving average for small datasets
                def moving_average(data, window_size):
                    cumsum = np.cumsum(np.insert(data, 0, 0)) 
                    return (cumsum[window_size:] - cumsum[:-window_size]) / window_size
                
                ma_window = min(window_size // 2, len(rsv_indices) // 2)
                if ma_window >= 2:
                    ma = moving_average(thresholds, ma_window)
                    pad = ma_window // 2
                    full_ma = np.pad(ma, (pad, pad), 'edge')[:len(thresholds)]
                    ax1.plot(rsv_indices, full_ma, color=color, linewidth=2.0, 
                            label=f'Class {class_idx}', alpha=class_line_opacity)
    
    # Add overall average for trained model
    overall_avg_trained = results['trained_model']['avg_thresholds']
    ax1.plot(rsv_indices, overall_avg_trained, color='black', linewidth=3.5, 
             label='Trained Model (Avg)', alpha=1.0, linestyle='-')
    
    # Add overall average for random model
    overall_avg_random = results['random_model']['avg_thresholds']
    ax1.plot(rsv_indices, overall_avg_random, color='grey', linewidth=3.5, 
             label='Random Model (Avg)', alpha=1.0, linestyle='-')
    
    # Set labels and title with larger font
    ax1.set_xlabel("RSV Index (ordered by singular value)", fontsize=MEDIUM_SIZE)
    ax1.set_ylabel("Max α with Cosine Similarity ≥ 0.95", fontsize=MEDIUM_SIZE)
    ax1.grid(alpha=0.3)
    
    # Create secondary y-axis for singular values
    ax2 = ax1.twinx()
    
    # Plot singular values on secondary y-axis for both models
    ax2.semilogy(rsv_indices, results['trained_model']['avg_singular_values'], 
                color='black', alpha=0.5, label='Trained Model σ', 
                linestyle='--', linewidth=2.0)
    
    ax2.semilogy(rsv_indices, results['random_model']['avg_singular_values'], 
                color='grey', alpha=0.5, label='Random Model σ', 
                linestyle='--', linewidth=2.0)
    
    ax2.set_ylabel("Singular Value (log scale)", fontsize=MEDIUM_SIZE)
    
    # Make tick labels larger
    ax1.tick_params(axis='both', which='major', labelsize=SMALL_SIZE)
    ax2.tick_params(axis='both', which='major', labelsize=SMALL_SIZE)
    
    # Combine legends with larger font
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=MEDIUM_SIZE, loc='center right', 
              ncol=2 if len(class_ids) > 5 else 1)
    
    plt.title("Comparison of Linear Approximation Thresholds Across Classes", fontsize=BIGGER_SIZE)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    
    # Reset font sizes to default after plotting
    plt.rcdefaults()
    
    plt.show()

In [ ]:
# plot_multiclass_comparison(
#     results,
#     output_path='overall_averages_comparison.png', 
#     show_class_lines=False,
#     show_raw_data=False
# )

In [ ]:
# plot_multiclass_comparison(
#     results,
#     output_path='all_classes_comparison.png', 
#     show_class_lines=True,
#     show_raw_data=False
# )

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST, CIFAR10
import numpy as np
import matplotlib.pyplot as plt
import yaml
import os
import importlib

def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

def get_model(model_config):
    """Dynamically import and instantiate the model based on config."""
    model_name = model_config['name']
    model_args = model_config.get('args', {})
    
    # Import the module containing the model
    module_path = model_config.get('module', 'src.models')
    
    # Use the 'file' field from config if available, otherwise fallback to lowercased model name
    model_file = model_config.get('file', model_name.lower())
    
    # Import the module
    module = importlib.import_module(f"{module_path}.{model_file}")
    
    # Get the model class and instantiate it
    model_class = getattr(module, model_name)
    return model_class(**model_args)

In [ ]:
def analyze_multiple_samples(random_checkpoint_path, trained_checkpoint_path, config_path, 
                            device='cuda', num_rsv_to_analyze=100, 
                            samples_per_class=5, class_count=10, rsv_scale=56.0):
    """
    Analyze linear approximation thresholds for multiple samples from each class.
    Only select samples that are correctly classified by the trained model.
    
    Args:
        random_checkpoint_path: Path to random initialization checkpoint
        trained_checkpoint_path: Path to trained model checkpoint
        config_path: Path to the model configuration YAML file
        device: Device to run computations on
        num_rsv_to_analyze: Number of RSVs to analyze
        samples_per_class: Number of samples to analyze per class
        class_count: Number of classes in dataset
    """
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load configuration
    config = load_config(config_path)
    
    # Load models using the configuration
    random_model = get_model(config['model']).to(device)
    trained_model = get_model(config['model']).to(device)
    
    # Load weights
    random_model.load_state_dict(torch.load(random_checkpoint_path, map_location=device))
    trained_model.load_state_dict(torch.load(trained_checkpoint_path, map_location=device))
    
    # Set models to evaluation mode
    random_model.eval()
    trained_model.eval()
    
    # Load dataset based on config
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            config['dataset']['transform']['normalize']['mean'],
            config['dataset']['transform']['normalize']['std']
        )
    ])
    
    # Load the dataset specified in the config
    if config['dataset']['name'].lower() == 'mnist':
        train_dataset = MNIST(root='../data/MNIST-Original', train=True, transform=transform, download=True)
    elif config['dataset']['name'].lower() == 'cifar10':
        train_dataset = CIFAR10(root='../data/CIFAR10', train=True, transform=transform, download=True)
    else:
        raise ValueError(f"Unsupported dataset: {config['dataset']['name']}")
    
    # Create DataLoader to iterate through dataset
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)
    
    # Dictionary to store correctly classified samples for each class
    class_samples = {i: [] for i in range(class_count)}
    
    # Collect correctly classified samples for each class
    print("Collecting correctly classified samples for each class...")
    with torch.no_grad():
        for image, label in tqdm(train_loader):
            # Get the true label
            true_label = label.item()
            
            # Move image to device
            image = image.to(device)
            
            # Get the model's prediction
            output = trained_model(image)
            pred_label = torch.argmax(output, dim=1).item()
            
            # Check if the prediction matches the true label
            if pred_label == true_label and len(class_samples[true_label]) < samples_per_class:
                # Keep the correctly classified sample
                class_samples[true_label].append(image.squeeze(0).cpu())
            
            # Check if we have enough samples for all classes
            if all(len(samples) >= samples_per_class for samples in class_samples.values()):
                break
    
    # Check if we collected enough samples for each class
    for class_idx, samples in class_samples.items():
        if len(samples) < samples_per_class:
            print(f"Warning: Could only collect {len(samples)} correctly classified samples for class {class_idx}")
    
    # Initialize result storage
    results = {
        'rsv_indices': None,
        'random_model': {
            'thresholds_by_class': {i: [] for i in range(class_count)},
            'singular_values_by_class': {i: [] for i in range(class_count)},
            'avg_thresholds': None,
            'avg_singular_values': None
        },
        'trained_model': {
            'thresholds_by_class': {i: [] for i in range(class_count)},
            'singular_values_by_class': {i: [] for i in range(class_count)},
            'avg_thresholds': None,
            'avg_singular_values': None
        }
    }
    
    # Analyze each class separately
    for class_idx in tqdm(range(class_count), desc="Analyzing classes"):
        # Skip classes with no samples
        if len(class_samples[class_idx]) == 0:
            continue
            
        # Analyze each sample for this class
        for sample_idx, sample in enumerate(class_samples[class_idx]):
            # Compute Jacobian for the sample
            jacobian_random = compute_jacobian(random_model, sample.unsqueeze(0).to(device))
            jacobian_trained = compute_jacobian(trained_model, sample.unsqueeze(0).to(device))
            
            # Compute SVD
            _, S_random, V_random = compute_svd(jacobian_random)
            _, S_trained, V_trained = compute_svd(jacobian_trained)
            
            # Compute thresholds
            rsv_indices_random, thresholds_random, sv_values_random = compute_linear_approximation_thresholds(
                random_model, sample, V_random, S_random, 
                device=device, num_rsv_to_analyze=num_rsv_to_analyze, rsv_scale=rsv_scale
            )
            
            rsv_indices_trained, thresholds_trained, sv_values_trained = compute_linear_approximation_thresholds(
                trained_model, sample, V_trained, S_trained, 
                device=device, num_rsv_to_analyze=num_rsv_to_analyze, rsv_scale=rsv_scale
            )
            
            # Store results
            if results['rsv_indices'] is None:
                results['rsv_indices'] = rsv_indices_random  # Should be the same for all samples
            
            results['random_model']['thresholds_by_class'][class_idx].append(thresholds_random)
            results['random_model']['singular_values_by_class'][class_idx].append(sv_values_random)
            
            results['trained_model']['thresholds_by_class'][class_idx].append(thresholds_trained)
            results['trained_model']['singular_values_by_class'][class_idx].append(sv_values_trained)
    
    # Compute averages for each class
    random_avg_thresholds_by_class = {}
    random_avg_sv_by_class = {}
    trained_avg_thresholds_by_class = {}
    trained_avg_sv_by_class = {}
    
    for class_idx in range(class_count):
        # Skip classes with no samples
        if len(results['random_model']['thresholds_by_class'][class_idx]) == 0:
            continue
            
        # Stack all samples for this class
        random_thresholds_stack = np.stack(results['random_model']['thresholds_by_class'][class_idx])
        random_sv_stack = np.stack(results['random_model']['singular_values_by_class'][class_idx])
        
        trained_thresholds_stack = np.stack(results['trained_model']['thresholds_by_class'][class_idx])
        trained_sv_stack = np.stack(results['trained_model']['singular_values_by_class'][class_idx])
        
        # Compute averages
        random_avg_thresholds_by_class[class_idx] = np.mean(random_thresholds_stack, axis=0)
        random_avg_sv_by_class[class_idx] = np.mean(random_sv_stack, axis=0)
        # random_avg_sv_by_class[class_idx] = np.median(random_sv_stack, axis=0) #np.mean(random_sv_stack, axis=0)
        # random_avg_thresholds_by_class[class_idx] = np.median(random_thresholds_stack, axis=0) #np.mean(random_thresholds_stack, axis=0)
        
        # trained_avg_thresholds_by_class[class_idx] = np.median(trained_thresholds_stack, axis=0) #np.mean(trained_thresholds_stack, axis=0)
        # trained_avg_sv_by_class[class_idx] = np.median(trained_sv_stack, axis=0) #np.mean(trained_sv_stack, axis=0)
        trained_avg_thresholds_by_class[class_idx] = np.mean(trained_thresholds_stack, axis=0)
        trained_avg_sv_by_class[class_idx] = np.mean(trained_sv_stack, axis=0)
    
    # Compute overall averages across all classes with samples
    valid_classes = [i for i in range(class_count) if i in random_avg_thresholds_by_class]
    
    if valid_classes:
        random_all_thresholds = np.stack([random_avg_thresholds_by_class[i] for i in valid_classes])
        random_all_sv = np.stack([random_avg_sv_by_class[i] for i in valid_classes])
        
        trained_all_thresholds = np.stack([trained_avg_thresholds_by_class[i] for i in valid_classes])
        trained_all_sv = np.stack([trained_avg_sv_by_class[i] for i in valid_classes])
        
        results['random_model']['avg_thresholds'] = np.mean(random_all_thresholds, axis=0)
        results['random_model']['avg_singular_values'] = np.mean(random_all_sv, axis=0)

        # results['random_model']['avg_thresholds'] = np.median(random_all_thresholds, axis=0) #np.mean(random_all_thresholds, axis=0)
        # results['random_model']['avg_singular_values'] = np.median(random_all_sv, axis=0) #np.mean(random_all_sv, axis=0)
        
        results['trained_model']['avg_thresholds'] = np.mean(trained_all_thresholds, axis=0)
        results['trained_model']['avg_singular_values'] = np.mean(trained_all_sv, axis=0)
        # results['trained_model']['avg_thresholds'] = np.median(trained_all_thresholds, axis=0) #np.mean(trained_all_thresholds, axis=0)
        # results['trained_model']['avg_singular_values'] = np.median(trained_all_sv, axis=0) #np.mean(trained_all_sv, axis=0)

    else:
        print("Warning: No valid classes with samples found.")
    
    # Save all the per-class averages too
    results['random_model']['avg_thresholds_by_class'] = random_avg_thresholds_by_class
    results['random_model']['avg_singular_values_by_class'] = random_avg_sv_by_class
    results['trained_model']['avg_thresholds_by_class'] = trained_avg_thresholds_by_class
    results['trained_model']['avg_singular_values_by_class'] = trained_avg_sv_by_class
    
    # Extract model type for naming output files
    model_type = config['model']['name'].lower()
    
    return results

# MNIST

## MLP - ALL

In [ ]:
results_mlp_mnist = analyze_multiple_samples(
    random_checkpoint_path="checkpoints/mnist_original_mlp/model_epoch0_batch0.pth",
    trained_checkpoint_path="checkpoints/mnist_original_mlp/best_model_mnist.pth",
    config_path="configs/training/mnist_mlp.yaml",
    device="cuda",
    num_rsv_to_analyze=100,
    samples_per_class=5,
    class_count=10,
    rsv_scale=56.0
)

In [ ]:
save_results(results_mlp_mnist, 'mnist_mlp_analysis_results.pkl')

In [ ]:
results_mlp_mnist = load_results('mnist_mlp_analysis_results.pkl')
plot_multiclass_comparison(
    results_mlp_mnist,
    output_path='mnist_mlp_analysis_results_all_classes_comparison.png', 
    show_class_lines=False,
    show_raw_data=False
)

## CNN - ALL

In [ ]:
# Random initialization vs trained model for CNN on MNIST
results_cnn_mnist = analyze_multiple_samples(
    random_checkpoint_path="checkpoints/mnist_original_cnn/model_epoch0_batch0.pth",
    trained_checkpoint_path="checkpoints/mnist_original_cnn/best_model_mnist.pth",
    config_path="configs/training/mnist_cnn.yaml",
    device="cuda",
    num_rsv_to_analyze=100,
    samples_per_class=5,
    class_count=10,
    rsv_scale=56.0
)

In [ ]:
save_results(results_cnn_mnist, 'mnist_cnn_analysis_results.pkl')

In [ ]:
results_cnn_mnist = load_results('mnist_cnn_analysis_results.pkl')
plot_multiclass_comparison(
    results_cnn_mnist,
    output_path='mnist_cnn_analysis_results_all_classes_comparison.png', 
    show_class_lines=True,
    show_raw_data=False
)

## VIT - ALL

In [ ]:
# Random initialization vs trained model for CNN on MNIST
results_vit_mnist = analyze_multiple_samples(
    random_checkpoint_path="checkpoints/mnist_original_vit/model_epoch0_batch0.pth",
    trained_checkpoint_path="checkpoints/mnist_original_vit/best_model_mnist.pth",
    config_path="configs/training/mnist_vit.yaml",
    device="cuda",
    num_rsv_to_analyze=100,
    samples_per_class=5,
    class_count=10,
    rsv_scale=56.0
)

In [ ]:
save_results(results_vit_mnist, 'mnist_vit_analysis_results.pkl')

In [ ]:
results_vit_mnist = load_results('mnist_vit_analysis_results.pkl')
plot_multiclass_comparison(
    results_vit_mnist,
    output_path='mnist_vit_analysis_results_all_classes_comparison.png', 
    show_class_lines=True,
    show_raw_data=False
)

# CIFAR 10

# MLP 784

In [ ]:
results_mlp_cifar10 = analyze_multiple_samples(
    random_checkpoint_path="checkpoints/cifar10_mlp/model_epoch0_batch0.pth",
    trained_checkpoint_path="checkpoints/cifar10_mlp/best_model_cifar10.pth",
    config_path="configs/training/cifar10_mlp.yaml",
    device="cuda",
    num_rsv_to_analyze=100,
    samples_per_class=5,
    class_count=10,
    rsv_scale=111.0
)

In [ ]:
save_results(results_mlp_cifar10, 'cifar10_mlp_analysis_results.pkl')

In [ ]:
results_mlp_cifar10 = load_results('cifar10_mlp_analysis_results.pkl')
plot_multiclass_comparison(
    results_mlp_cifar10,
    output_path='cifar10_mlp_analysis_results_all_classes_comparison.png', 
    show_class_lines=True,
    show_raw_data=False
)

## CNN 784

In [ ]:
results_cnn_cifar10 = analyze_multiple_samples(
    random_checkpoint_path="checkpoints/cifar10_cnn/model_epoch0_batch0.pth",
    trained_checkpoint_path="checkpoints/cifar10_cnn/best_model_cifar10.pth",
    config_path="configs/training/cifar10_cnn.yaml",
    device="cuda",
    num_rsv_to_analyze=100,
    samples_per_class=5,
    class_count=10,
    rsv_scale=111.0
)

In [ ]:
save_results(results_cnn_cifar10, 'cifar10_cnn_analysis_results.pkl')

In [ ]:
results_cnn_cifar10 = load_results('cifar10_cnn_analysis_results.pkl')
plot_multiclass_comparison(
    results_cnn_cifar10,
    output_path='cifar10_cnn_analysis_results_all_classes_comparison.png', 
    show_class_lines=True,
    show_raw_data=False
)

## VIT 784

In [ ]:
results_vit_cifar10 = analyze_multiple_samples(
    random_checkpoint_path="checkpoints/cifar10_vit/model_epoch0_batch0.pth",
    trained_checkpoint_path="checkpoints/cifar10_vit/best_model_cifar10.pth",
    config_path="configs/training/cifar10_vit.yaml",
    device="cuda",
    num_rsv_to_analyze=100,
    samples_per_class=5,
    class_count=10,
    rsv_scale=111.0
)

In [ ]:
save_results(results_vit_cifar10, 'cifar10_vit_analysis_results.pkl')

In [ ]:
results_vit_cifar10 = load_results('cifar10_vit_analysis_results.pkl')
plot_multiclass_comparison(
    results_vit_cifar10,
    output_path='cifar10_vit_analysis_results_all_classes_comparison.png', 
    show_class_lines=True,
    show_raw_data=False
)